In [1]:
import torch

# GPUの確認
print("CUDA Available:", torch.cuda.is_available())
print("Device Count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("Current Device:", torch.cuda.current_device())
    print("Device Name:", torch.cuda.get_device_name(torch.cuda.current_device()))

CUDA Available: True
Device Count: 1
Current Device: 0
Device Name: Quadro RTX 6000


In [ ]:
from ultralytics import YOLO

# モデルのロード
model = YOLO('640m.pt')  # 提供された .pt モデルをロード

# トレーニングの実行
model.train(data='data.yaml', epochs=50, batch=16, imgsz=640, verbose=False)

# ファインチューニング後のモデルを保存
model.save('640m_finetuned.pt')
model.export(format='onnx', imgsz=640)  # imgszはモデルの入力サイズ

Ultralytics 8.3.59 🚀 Python-3.10.12 torch-2.5.1+cu124 CUDA:0 (Quadro RTX 6000, 24205MiB)
engine/trainer: task=detect, mode=train, model=640m.pt, data=data.yaml, epochs=50, time=None, patience=100, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=train3, exist_ok=False, pretrained=True, optimizer=auto, verbose=False, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=False, save_crop=False, show_labels=True, show_conf=True, show_boxes=True, line

train: Scanning /workspace/datasets/labels/train.cache... 1665 images, 0 backgrounds, 0 corrupt: 100%|██████████| 1665/1665 [00:00<?, ?it/s]
val: Scanning /workspace/datasets/labels/val.cache... 448 images, 0 backgrounds, 0 corrupt: 100%|██████████| 448/448 [00:00<?, ?it/s]


Plotting labels to runs/detect/train3/labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.000455, momentum=0.9) with parameter groups 77 weight(decay=0.0), 84 weight(decay=0.0005), 83 bias(decay=0.0)
Image sizes 640 train, 640 val
Using 8 dataloader workers
Logging results to runs/detect/train3
Starting training for 50 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


       1/50      6.95G      1.785      1.773      1.651          0        640: 100%|██████████| 105/105 [00:25<00:00,  4.16it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 14/14 [00:03<00:00,  4.56it/s]

                   all        448        799      0.696      0.606      0.641      0.359



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


       2/50      7.32G      1.297     0.9103      1.279         55        640:  48%|████▊     | 50/105 [00:11<00:12,  4.32it/s]

In [17]:

%matplotlib inline
import cv2
import glob
import matplotlib.pyplot as plt

# ファイル一覧を取得
files = glob.glob("datasets/images/val/*.jpg")
files = glob.glob("datasets/*.jpg")
files = []

# 各画像に対して推論と描画
for file in files:
    # 推論
    results = model.predict(file, conf=0.25)

    # 結果からbboxを取得
    boxes = results[0].boxes.xyxy  # bboxの座標 (x_min, y_min, x_max, y_max)
    scores = results[0].boxes.conf  # 各bboxの信頼度
    classes = results[0].boxes.cls  # 各bboxのクラスID

    # 元画像を読み込み (RGBに変換)
    image = cv2.imread(file)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)  # BGR → RGB

    # 描画
    for i, box in enumerate(boxes):
        x_min, y_min, x_max, y_max = map(int, box)  # 座標を整数に変換
        class_id = int(classes[i])  # クラスID
        score = scores[i]  # 信頼度
        label = f"{results[0].names[class_id]} {score:.2f}"  # ラベル表示 (例: class_14 0.89)

        # バウンディングボックスを描画
        cv2.rectangle(image, (x_min, y_min), (x_max, y_max), (0, 255, 0), 2)  # 緑色の枠線
        # ラベルを描画
        cv2.putText(image, label, (x_min, y_min - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

    # Jupyter Notebook 上で画像を表示
    plt.figure(figsize=(10, 10))
    plt.imshow(image)
    plt.axis("off")  # 軸を非表示
    plt.title(file)  # ファイル名をタイトルとして表示
    plt.show()
